# Orientation as Holonomy of the Reader Configuration Space — companion notebook

Dean A. Kulik, QuHarmonics Research Group (ORCID 0009-0003-3128-8828), September 2026

Reproduces every number in the paper. Standard library only, no files read, no downloads.
Fixed seeds throughout. Runs in about a minute.

In [1]:
import itertools, random
from collections import defaultdict, deque, Counter
from fractions import Fraction

def nbrs(c, dims):
    out=[]
    for i in range(len(dims)):
        for d in (1,-1):
            v=list(c); v[i]=(v[i]+d)%dims[i]; out.append(tuple(v))
    return out

def connected(S, dims):
    if not S: return False
    S=set(S); s=next(iter(S)); seen={s}; q=deque([s])
    while q:
        u=q.popleft()
        for v in nbrs(u,dims):
            if v in S and v not in seen: seen.add(v); q.append(v)
    return seen==S

def components(verts, adj):
    seen=set(); c=0
    for v in verts:
        if v in seen: continue
        c+=1; q=deque([v]); seen.add(v)
        while q:
            u=q.popleft()
            for w in adj[u]:
                if w not in seen: seen.add(w); q.append(w)
    return c
print('loaded')

loaded


## Proposition 3.1 — dim of the conserved channel equals the cycle space of $K_m$

In [2]:
def rank(rows):
    rows=[list(map(Fraction,r)) for r in rows]; R=[]
    for r in rows:
        for b in R:
            p=next((k for k,v in enumerate(b) if v!=0), None)
            if p is not None and r[p]!=0:
                f=r[p]/b[p]; r=[a-f*c for a,c in zip(r,b)]
        if any(v!=0 for v in r): R.append(r)
    return len(R)

print(f"{'m':>3} {'(m-1)(m-2)/2':>14} {'|E(K_m)|-(m-1)':>16} {'span of triangles':>19}")
for m in range(2,9):
    E=[(i,j) for i in range(m) for j in range(i+1,m)]; idx={e:k for k,e in enumerate(E)}
    tris=[]
    for a,b,c in itertools.combinations(range(m),3):
        v=[0]*len(E); v[idx[(a,b)]]+=1; v[idx[(b,c)]]+=1; v[idx[(a,c)]]-=1
        tris.append(v)
    print(f"{m:>3} {(m-1)*(m-2)//2:>14} {len(E)-(m-1):>16} {(rank(tris) if tris else 0):>19}")

  m   (m-1)(m-2)/2   |E(K_m)|-(m-1)   span of triangles
  2              0                0                   0
  3              1                1                   1
  4              3                3                   3
  5              6                6                   6
  6             10               10                  10
  7             15               15                  15
  8             21               21                  21


## Proposition 3.2 — the channel is irreducible, and equals the sign rep exactly at m = 3

In [3]:
def sign(s):
    n=len(s); sg=1; seen=[False]*n
    for i in range(n):
        if not seen[i]:
            L=0;j=i
            while not seen[j]: seen[j]=True; L+=1; j=s[j]
            if L%2==0: sg=-sg
    return sg

def channel_module(m):
    E=[(i,j) for i in range(m) for j in range(i+1,m)]; idx={e:k for k,e in enumerate(E)}
    def act(s,v):
        w=[Fraction(0)]*len(E)
        for (i,j),k in idx.items():
            a,b=s[i],s[j]
            w[k]= v[idx[(a,b)]] if a<b else -v[idx[(b,a)]]
        return w
    B=[]
    for a,b,c in itertools.combinations(range(m),3):
        v=[Fraction(0)]*len(E); v[idx[(a,b)]]+=1; v[idx[(b,c)]]+=1; v[idx[(a,c)]]-=1
        keep=[]
        for x in B+[v]:
            pass
        B.append(v)
    # reduce to a basis
    basis=[]
    for v in B:
        if rank([list(map(int,x)) for x in basis+[v]])>len(basis): basis.append(v)
    G=list(itertools.permutations(range(m)))
    proj=[]
    for v in basis:
        acc=[Fraction(0)]*len(E)
        for s in G:
            w=act(s,v); sg=sign(s)
            acc=[a+sg*Fraction(x,len(G)) for a,x in zip(acc,w)]
        proj.append(acc)
    return len(basis), rank([[int(x*len(G)) for x in p] for p in proj])

print(f"{'m':>3} {'dim':>5} {'sign-isotypic dim':>19}")
for m in (3,4,5,6):
    d,s=channel_module(m)
    print(f"{m:>3} {d:>5} {s:>19}")
print()
print('sign representation appears exactly at m = 3')

  m   dim   sign-isotypic dim
  3     1                   1
  4     3                   0
  5     6                   0


  6    10                   0

sign representation appears exactly at m = 3


## Proposition 4.1 — the sign is not a function of T

One permutation, one unordered partition, six labelings.

In [4]:
def table(p, lab, m=3):
    T=[[0]*m for _ in range(m)]
    for x,y in enumerate(p): T[lab[x]][lab[y]]+=1
    return T
def anti(T): return [[T[i][j]-T[j][i] for j in range(3)] for i in range(3)]

N=6; T=(1,2,0,4,5,3); blocks=[{0,3},{1,4},{2,5}]
print(f"T = {T}   partition {{0,3}}|{{1,4}}|{{2,5}}")
print(f"  {'labeling':>12} {'a':>5} {'|a|':>5} {'sgn(a)':>8}")
vals=[]
for perm in itertools.permutations(range(3)):
    lab=[0]*N
    for k,bi in enumerate(perm):
        for x in blocks[bi]: lab[x]=k
    a=anti(table(T,lab))[0][1]; vals.append(a)
    print(f"  {str(perm):>12} {a:>5} {abs(a):>5} {(1 if a>0 else -1):>8}")
print(f"\n  |a| constant: {len(set(map(abs,vals)))==1}")
print(f"  sgn(a) takes both values: {len({(v>0)-(v<0) for v in vals})>1}")

T = (1, 2, 0, 4, 5, 3)   partition {0,3}|{1,4}|{2,5}
      labeling     a   |a|   sgn(a)
     (0, 1, 2)     2     2        1
     (0, 2, 1)    -2     2       -1
     (1, 0, 2)    -2     2       -1
     (1, 2, 0)     2     2        1
     (2, 0, 1)     2     2        1
     (2, 1, 0)    -2     2       -1

  |a| constant: True
  sgn(a) takes both values: True


## Theorem 5.1 — the exchange loop, general in k

In [5]:
def exchange_loop(k):
    A=frozenset(range(k)); B=frozenset(range(k,2*k)); C=frozenset(range(2*k,3*k))
    b,c=sorted(B),sorted(C); Bc,Cc=set(B),set(C); seq=[(A,B,C)]
    for i in range(k):
        Bc.discard(b[i]); Bc.add(c[i]); Cc.discard(c[i]); Cc.add(b[i])
        seq.append((A,frozenset(Bc),frozenset(Cc)))
    return seq

print(f"  {'k':>3} {'moves':>6} {'same partition':>16} {'is (B C)':>10}")
for k in (2,3,4,5,8):
    s=exchange_loop(k); a,b0,c0=s[0]; a2,b2,c2=s[-1]
    print(f"  {k:>3} {len(s)-1:>6} {str({a,b0,c0}=={a2,b2,c2}):>16} {str(b2==c0 and c2==b0):>10}")

    k  moves   same partition   is (B C)
    2      2             True       True
    3      3             True       True
    4      4             True       True
    5      5             True       True
    8      8             True       True


## Section 6 — classification of im rho

Exhaustive component counts where feasible; explicit loops otherwise.

In [6]:
def canon_cyc(o):
    i=next(j for j,b in enumerate(o) if min(min(x) if isinstance(x,frozenset) else x for x in [b])==min(min(y) for y in o))
    return o
def canon(o):
    # cyclic class: rotate so the block containing the globally smallest cell is first
    lo=min(min(b) for b in o)
    i=next(j for j,b in enumerate(o) if lo in b)
    return tuple(o[(i+j)%3] for j in range(3))

def equal_parts(N,k):
    out=[]
    def rec(rem,acc):
        rem=sorted(rem)
        if not rem:
            if len(acc)==3: out.append(tuple(acc))
            return
        if len(acc)==3: return
        f=rem[0]
        for c in itertools.combinations(rem[1:],k-1):
            b=frozenset((f,)+c); rec([x for x in rem if x not in b], acc+[b])
    rec(list(range(N)),[]); return out

def classify(name, cover, movefn):
    cs=set(cover); adj=defaultdict(set)
    for v in cover:
        for w in movefn(v):
            if w in cs: adj[v].add(w); adj[w].add(v)
    base=sorted({frozenset(v) for v in cover}); adjB=defaultdict(set)
    for v in cover:
        for w in adj[v]:
            adjB[frozenset(v)].add(frozenset(w)); adjB[frozenset(w)].add(frozenset(v))
    cb,cc=components(base,adjB),components(cover,adj)
    print(f"  {name:<44} base {cb:>3} cover {cc:>3}  im rho = {'A_3 (trivial sgn)' if cc==2*cb else 'S_3'}")

N=6
cover=sorted({canon(p) for b in equal_parts(N,2) for p in itertools.permutations(b)})
def m_any(v):
    out=[]
    for i,j in itertools.combinations(range(3),2):
        for x in v[i]:
            for y in v[j]:
                n=list(v); n[i]=frozenset((v[i]-{x})|{y}); n[j]=frozenset((v[j]-{y})|{x})
                out.append(canon(tuple(n)))
    return out
def m_adj(v):
    out=[]; where={}
    for i,b in enumerate(v):
        for x in b: where[x]=i
    for x in range(N):
        y=(x+1)%N; i,j=where[x],where[y]
        if i==j: continue
        n=list(v); n[i]=frozenset((v[i]-{x})|{y}); n[j]=frozenset((v[j]-{y})|{x})
        out.append(canon(tuple(n)))
    return out
classify('complete ambient, size-preserving swaps', cover, m_any)
classify('cycle ambient, adjacent-cell swaps only', cover, m_adj)

  complete ambient, size-preserving swaps      base   1 cover   1  im rho = S_3
  cycle ambient, adjacent-cell swaps only      base   1 cover   1  im rho = S_3


In [7]:
# contiguous arcs of an oriented cycle
def arc_cover(N):
    out=set()
    for c in itertools.combinations(range(N),3):
        cs=sorted(c); arcs=[]
        for i in range(3):
            s,e=cs[i],cs[(i+1)%3]
            arcs.append(frozenset((s+t)%N for t in range((e-s)%N)))
        if all(len(a)>0 for a in arcs) and len(set().union(*arcs))==N:
            for p in itertools.permutations(arcs): out.add(canon(p))
    return sorted(out)

for N in (6,9,12):
    cov=arc_cover(N); cs=set(cov)
    def slide(v, N=N, cs=cs):
        out=[]
        for i in range(3):
            j=(i+1)%3
            if len(v[i])<2: continue
            for x in v[i]:
                if (x+1)%N in v[j] or (x-1)%N in v[j]:
                    n=list(v); n[i]=frozenset(v[i]-{x}); n[j]=frozenset(v[j]|{x})
                    if all(len(b)>0 for b in n):
                        w=canon(tuple(n))
                        if w in cs: out.append(w)
        return out
    classify(f'cycle ambient N={N}, sectors are contiguous arcs', cov, slide)

  cycle ambient N=6, sectors are contiguous arcs base   1 cover   2  im rho = A_3 (trivial sgn)
  cycle ambient N=9, sectors are contiguous arcs base   1 cover   2  im rho = A_3 (trivial sgn)
  cycle ambient N=12, sectors are contiguous arcs base   1 cover   2  im rho = A_3 (trivial sgn)


In [8]:
# 2-torus, connected regions of any size
def torus_cells(n,m): return [(i,j) for i in range(n) for j in range(m)]
def all_connected_parts(n,m):
    cells=torus_cells(n,m); dims=(n,m); out=[]
    for k in range(1,len(cells)-1):
        for A in itertools.combinations(cells,k):
            if not connected(A,dims): continue
            rest=[c for c in cells if c not in A]
            for k2 in range(1,len(rest)):
                for Bb in itertools.combinations(rest,k2):
                    if not connected(Bb,dims): continue
                    C=tuple(c for c in rest if c not in Bb)
                    if C and connected(C,dims): out.append((frozenset(A),frozenset(Bb),frozenset(C)))
    return out

n=m=3; dims=(n,m)
parts=all_connected_parts(n,m)
cover2=sorted({canon(p) for b in parts for p in itertools.permutations(b)})
cs2=set(cover2)
def transfer(v):
    out=[]
    for i,j in itertools.permutations(range(3),2):
        if len(v[i])<2: continue
        for x in v[i]:
            if not any(y in v[j] for y in nbrs(x,dims)): continue
            ni=frozenset(v[i]-{x}); nj=frozenset(v[j]|{x})
            if connected(ni,dims) and connected(nj,dims):
                w=canon(tuple(nj if k==j else (ni if k==i else v[k]) for k in range(3)))
                if w in cs2: out.append(w)
    return out
classify('2-torus Z_3 x Z_3, connected regions', cover2, transfer)

  2-torus Z_3 x Z_3, connected regions         base   1 cover   1  im rho = S_3


### Fundamental-cycle census

Spanning tree, then the monodromy of each non-tree edge's fundamental cycle.
Counts are basis dependent; the generated subgroup and the presence of an odd
element are not.

In [9]:
def monodromy_census(cover, movefn, label):
    cs=set(cover); adj=defaultdict(set)
    for v in cover:
        for w in movefn(v):
            if w in cs: adj[v].add(w); adj[w].add(v)
    base=sorted({frozenset(v) for v in cover})
    bidx={b:i for i,b in enumerate(base)}
    badj=defaultdict(set)
    for v in cover:
        for w in adj[v]:
            badj[frozenset(v)].add(frozenset(w)); badj[frozenset(w)].add(frozenset(v))
    lift_from={}
    def lift(La,Q):
        for c in movefn(La):
            if frozenset(c)==Q: return c
        return None
    start=base[0]
    L={start: tuple(sorted(start,key=lambda b:sorted(b)))}
    parent={start:None}; q=deque([start])
    while q:
        Pp=q.popleft()
        for Q in badj[Pp]:
            if Q in L: continue
            c=lift(L[Pp],Q)
            if c is None: continue
            L[Q]=c; parent[Q]=Pp; q.append(Q)
    tree={(min(bidx[c],bidx[parent[c]]),max(bidx[c],bidx[parent[c]])) for c in parent if parent[c]}
    def perm(La,Lb): return tuple(Lb.index(La[k]) for k in range(3))
    rhos=[]
    for Pp in L:
        for Q in badj[Pp]:
            if Q not in L: continue
            key=(min(bidx[Pp],bidx[Q]),max(bidx[Pp],bidx[Q]))
            if key in tree or bidx[Pp]>bidx[Q]: continue
            c=lift(L[Pp],Q)
            if c is None: continue
            rhos.append(perm(c,list(L[Q])))
    nm={(0,1,2):'e',(1,2,0):'(123)',(2,0,1):'(132)',(0,2,1):'(23)',(2,1,0):'(13)',(1,0,2):'(12)'}
    V=len(L); E=sum(len(badj[b]&set(L)) for b in L)//2
    print(f"{label}: V={V} E={E} beta_1={E-V+1}, cycles measured {len(rhos)}")
    print(f"   {dict(Counter(nm[r] for r in rhos))}")
    print(f"   odd cycles: {sum(1 for r in rhos if sign(r)==-1)}")

# 1D: contiguous arcs of Z_6
cov1=arc_cover(6); cs1=set(cov1)
def slide6(v):
    out=[]; N=6
    for i in range(3):
        j=(i+1)%3
        if len(v[i])<2: continue
        for x in v[i]:
            if (x+1)%N in v[j] or (x-1)%N in v[j]:
                nn=list(v); nn[i]=frozenset(v[i]-{x}); nn[j]=frozenset(v[j]|{x})
                if all(len(b)>0 for b in nn):
                    w=canon(tuple(nn))
                    if w in cs1: out.append(w)
    return out
monodromy_census(cov1, slide6, '1D arcs, N=6')
print()
monodromy_census(cover2, transfer, '2D torus Z_3xZ_3')

1D arcs, N=6: V=20 E=36 beta_1=17, cycles measured 6
   {'e': 6}
   odd cycles: 0



2D torus Z_3xZ_3: V=986 E=5121 beta_1=4136, cycles measured 4136
   {'e': 2911, '(23)': 1225}
   odd cycles: 1225


## Theorem 5.1 under a connectivity constraint — 1D fails, 2D and 3D succeed

In [10]:
def try_exchange(dims, regions, budget=400):
    A,B,C=[set(r) for r in regions]; B0,C0=frozenset(B),frozenset(C); moves=0
    for _ in range(budget):
        if frozenset(B)==C0 and frozenset(C)==B0: return moves
        done=False
        for x in sorted(B-C0):
            for y in sorted(C-B0):
                nb=(B-{x})|{y}; nc=(C-{y})|{x}
                if connected(nb,dims) and connected(nc,dims):
                    B,C=nb,nc; moves+=1; done=True; break
            if done: break
        if not done: return None
    return None

print('1D cycle Z_6, three arcs of 2 :', try_exchange((6,), [{(0,),(1,)},{(2,),(3,)},{(4,),(5,)}]))
print('2D torus Z_3xZ_3, three rows  :', try_exchange((3,3),
      [{(0,j) for j in range(3)},{(1,j) for j in range(3)},{(2,j) for j in range(3)}]), 'moves')
print('3D torus Z_3^3, three slabs   :', try_exchange((3,3,3),
      [{(i,j,k) for j in range(3) for k in range(3)} for i in range(3)]), 'moves')
print()
print('A search that finds a loop is a proof; one that fails is not.')
print('The 1D claim rests on the exhaustive component count above, not on this failure.')

1D cycle Z_6, three arcs of 2 : None
2D torus Z_3xZ_3, three rows  : 3 moves
3D torus Z_3^3, three slabs   : 9 moves

A search that finds a loop is a proof; one that fails is not.
The 1D claim rests on the exhaustive component count above, not on this failure.


## Proposition 7.1 — a full channel with no triangle in the state graph

In [11]:
n=4; N=n*n
idx=lambda x,y:(x%n)*n+(y%n)
def torus_perm(dx,dy):
    p=[0]*N
    for x in range(n):
        for y in range(n): p[idx(x,y)]=idx(x+dx,y+dy)
    return tuple(p)
A_,B_=torus_perm(1,0),torus_perm(0,1)
E=set()
for p in (A_,B_):
    for x,y in enumerate(p):
        if x!=y: E.add(tuple(sorted((x,y))))
def has_tri(E,N):
    for a,b in E:
        for c in range(N):
            if c not in (a,b) and tuple(sorted((a,c))) in E and tuple(sorted((b,c))) in E: return True
    return False
print(f"Z_4 x Z_4: |V|={N} |E|={len(E)} cycle rank={len(E)-N+1} triangle={has_tri(E,N)}")
print(f"A and B commute: {tuple(A_[B_[i]] for i in range(N))==tuple(B_[A_[i]] for i in range(N))}")

lab=[((x//n)+(x%n))%3 for x in range(N)]
Tb=[[0]*3 for _ in range(3)]
for x,y in enumerate(A_): Tb[lab[x]][lab[y]]+=1
Am=[[Tb[i][j]-Tb[j][i] for j in range(3)] for i in range(3)]
print(f"sector sizes {sorted(Counter(lab).values())}, C={Tb}")
print(f"A_12={Am[0][1]} A_13={Am[0][2]} A_23={Am[1][2]}")

rng=random.Random(0); hits=0; trials=2000
for _ in range(trials):
    l=[rng.randrange(3) for _ in range(N)]
    if len(set(l))<3: continue
    t=[[0]*3 for _ in range(3)]
    for x,y in enumerate(A_): t[l[x]][l[y]]+=1
    if any(t[i][j]!=t[j][i] for i in range(3) for j in range(3)): hits+=1
print(f"random 3-sector partitions with nonzero channel: {hits}/{trials}")

Z_4 x Z_4: |V|=16 |E|=32 cycle rank=17 triangle=False
A and B commute: True
sector sizes [5, 5, 6], C=[[2, 4, 0], [0, 1, 4], [4, 0, 1]]
A_12=4 A_13=-4 A_23=4
random 3-sector partitions with nonzero channel: 1283/2000
